#  🚀  **Aircheck Workshop: Data Preprocessing & Analysis** 🧬
Welcome to the **Aircheck Hackaton**!

In this notebook, we will preprocess molecular data and extract meaningful features for machine learning.

---

## **Train Data: DEL-Based Compound Screening on WDR91 Protein**

**Reference:** https://pubmed.ncbi.nlm.nih.gov/35535861/

---

## **Test Data: ASMS-Based Compound Screening on WDR91 Protein**

**Reference:** https://www.biorxiv.org/content/10.1101/2025.01.17.633682.abstract

---

## **Target protein: WDR91**

WDR91 is a WD40 repeat-containing protein. Proteins in this family act as scaffolds for other proteins to stick to when forming multiprotein complexes WDR domain-containing proteins comprise one of the largest protein families in humans and are involved in a diverse array of cellular networks and diseases.

**look it up on uniprot:** https://www.uniprot.org/uniprotkb/A4D1P6/entry

WDR91 plays a critical role in regulating early-to-late endosomal maturation and trafficking, processes essential for cellular functions such as nutrient uptake, signal transduction, and membrane protein recycling.

**Reference:** https://www.nature.com/articles/nrd.2017.179

▶ **Actives**:  Label= 1      
▶ **Inactives**: Label = 0

**Reference:** Protein–Ligand Docking in the Machine-Learning Era: https://www.mdpi.com/1420-3049/27/14/4568

---

## **RDKit**
RDKit is an open-source cheminformatics toolkit written in C++ with Python bindings. features:
Molecular Representation: Handles molecules using formats like SMILES and InChI.

- **Molecular Descriptors**: Computes features like molecular weight and LogP.
- **Substructure Search**: Identifies specific molecular fragments or patterns.
- **Fingerprints**: Generates molecular fingerprints for similarity and clustering.
- **Visualization**: Creates 2D molecular drawings, integrates with Jupyter.
- **Chemical Reactions**: Simulates reactions and predicts products.
- **3D Operations**: Builds 3D structures and optimizes geometries.

---

## **🟢 1. Install and Import Dependencies**
In this section, we install the necessary packages for chemical data processing and machine learning.bold text

In [ ]:
# --- run-time tracking -----------------------------------------------------
# Times every cell, so the last cell can report how long the whole notebook took.
# Harmless outside Jupyter/Colab, and costs nothing to run.
import time as _time

ECHO_CELL_TIME = False    # True prints each cell's own time under its output

CELL_TIMES = []
_timer_state = {}

try:
    from IPython import get_ipython

    def _timer_pre(info):
        _timer_state["t0"] = _time.perf_counter()
        _timer_state["src"] = getattr(info, "raw_cell", "")

    def _timer_post(result):
        t0 = _timer_state.pop("t0", None)
        if t0 is None:
            return
        src = _timer_state.pop("src", "")
        first = next((l.strip() for l in src.split("\n") if l.strip()), "")
        taken = _time.perf_counter() - t0
        CELL_TIMES.append((taken, first[:70]))
        if ECHO_CELL_TIME:
            print(f"[cell {len(CELL_TIMES):>2}  {taken:6.2f}s]")

    _ip = get_ipython()
    if _ip is not None and not _timer_state.get("registered"):
        _ip.events.register("pre_run_cell", _timer_pre)
        _ip.events.register("post_run_cell", _timer_post)
        _timer_state["registered"] = True
except Exception:
    pass          # timing is a convenience; never let it break the notebook

NOTEBOOK_STARTED = _time.time()
# ---------------------------------------------------------------------------

# This notebook runs in Google Colab, on Databricks, and from a local clone of the
# repository. It uses the repository if it is already on disk and clones it if not,
# so the paths below come out the same in all three places.
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/ShagReza/Aircheck-Workshop-2026.git"
REPO_NAME = "Aircheck-Workshop-2026"

IN_COLAB = "google.colab" in sys.modules
IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ


def find_repo_root(start):
    """Walk up from `start` looking for the repository root, or None."""
    candidate = Path(start).resolve()
    while candidate != candidate.parent:
        if (candidate / "requirements.txt").exists():
            return candidate
        candidate = candidate.parent
    return None


# Already on disk? A local clone, or a Databricks Git folder, will be found here.
REPO_ROOT = find_repo_root(Path.cwd())

if REPO_ROOT is None:
    # Colab, or a runtime where the repository is not checked out: clone it.
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    REPO_ROOT = Path(REPO_NAME).resolve()

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

environment = "Colab" if IN_COLAB else "Databricks" if IN_DATABRICKS else "locally"
print(f"Running in {environment}")
print(f"Repository root: {REPO_ROOT}")
print(f"Data files:      {sorted(p.name for p in DATA_DIR.glob('*.parquet'))}")

In [ ]:
# Install every package the workshop needs, as listed in requirements.txt.
# In Colab and Databricks, install into the current runtime.
# Locally, assume the virtual environment has already been prepared.

import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
IN_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

if IN_COLAB or IN_DATABRICKS:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(REPO_ROOT / "requirements.txt"),
        ],
        check=True,
    )
    print("Requirements installed.")
else:
    print("Local run - install requirements with:")
    print(f"    python -m pip install -r {REPO_ROOT / 'requirements.txt'}")

In [ ]:
# Import libraries:
import pandas as pd
import numpy as np
import rdkit
from rdkit.Chem import MolFromSmiles
from rdkit.Chem import AllChem
import os
import matplotlib.pyplot as plt
import seaborn as sns
import umap

## 🟢 **2. Loading the Workshop Data**

The datasets ship with this repository, so there is nothing to download and no Google Drive
to mount. The bootstrap cell above already cloned the repository (in Colab) or found it on
disk (locally), so the files are ready to read from `data/`.

| File | Compounds | What it is |
|---|---|---|
| `data/sample-train.parquet` | 4,000 | DEL screen against WDR91, balanced 50/50 on `LABEL`. We train on this. |
| `data/sample-test.parquet` | 5,000 | Labelled evaluation set with `SMILES`. Only **9** compounds are active. |
| `data/sample-screen.parquet` | 5,000 | Compounds with `SMILES` and no label. These are what we screen and nominate. |

These are samples of the full AIRCHECK datasets, small enough to keep in the repository so
every participant starts from exactly the same data. The full datasets are available from
the AIRCHECK website: https://www.aircheck.ai/datasets

**A note on the fingerprint columns:** each fingerprint (`ECFP4`, `MACCS`, ...) is stored as
an **array of counts**, not as a comma-separated string. Reading a column gives you NumPy
arrays directly, so no parsing is needed.

In [ ]:
# Read the datasets into Pandas DataFrames.
df_train = pd.read_parquet(DATA_DIR / "sample-train.parquet")     # DEL screen, balanced, labelled
df_test = pd.read_parquet(DATA_DIR / "sample-test.parquet")       # labelled, with SMILES, 9 actives
df_screen = pd.read_parquet(DATA_DIR / "sample-screen.parquet")   # compounds to screen, with SMILES

for name, frame in [("df_train", df_train), ("df_test", df_test),
                    ("df_screen", df_screen)]:
    actives = int(frame["LABEL"].sum()) if "LABEL" in frame.columns else None
    print(f"{name:<10} {frame.shape[0]:>5} rows x {frame.shape[1]:>2} columns"
          + (f"   actives: {actives}" if actives is not None else "   (unlabelled)"))

# Each fingerprint cell is already a NumPy array of counts:
example = df_train["ECFP4"].iloc[0]
print(f"\nECFP4 for the first compound: {type(example).__name__} of {example.dtype}, "
      f"length {example.size}, {int((example > 0).sum())} non-zero bits")

---

## 🟢 **3. Exploring Train dataset**

In [ ]:
# Summary of the trian dataset
df_train.info()

# Columns' names
column_list = df_train.columns.tolist()
print("\nColumns' names: ", column_list)
print("\n")

# First 3 samples of the Test dataset
df_train.head(3)

In [ ]:
# Anomaly Detection

# Any Nulls?
null_counts = df_train.isnull().sum()
print(", ".join(f"{col}: {count}" for col, count in null_counts.items()))
print()
print("There are no nulls!" if null_counts.sum() == 0
      else ", ".join(f"{col}: {count}" for col, count in null_counts.items() if count > 0))

# Any Duplicates?
# The fingerprint columns hold NumPy arrays, which pandas cannot hash, so we check for
# duplicates on the scalar columns only.
scalar_cols = [c for c in df_train.columns if not isinstance(df_train[c].iloc[0], np.ndarray)]
duplicate_counts = df_train.duplicated(subset=scalar_cols).sum()
print("There are no duplicates!" if duplicate_counts == 0 else f"Total Duplicates: {duplicate_counts}")

**Let's check it out:**
* How many compounds are in this screen?
* How many of these are enriched?

In [ ]:
rows, cols = df_train.shape
print(f"Rows: {rows}, Columns: {cols}")

# Count occurrences of LABEL = 0 and LABEL = 1
count_not_enriched = (df_train['LABEL'] == 0).sum()
count_enriched = (df_train['LABEL'] == 1).sum()

# Print results
print(f"Compounds not enriched (LABEL = 0): {count_not_enriched}")
print(f"Compounds enriched (LABEL = 1): {count_enriched}")

Take some time to explore what the different columns mean:


| Field Name   | Data Type | Mode     | Description                                                                 |
|--------------|-----------|----------|-----------------------------------------------------------------------------|
| ID           | Long      | Required | Unique computed ID from DEL_ID concatenating DEL Library ID and Building Block IDs. |
| DEL_ID       | String    | Required | Unique ID of full enumerated DNA-Encoded Library (DEL) compound.            |
| LABEL        | Integer   | Required | Binary classification label for observed enrichment (0 not enriched, 1 enriched). |
| RawCount     | Integer   | Nullable | The sequence count (enrichment) for the specific target.                    |
| TARGET_ID    | String    | Required | Unique ID of the target.                                                   |
| ECFP4        | Bytes     | Nullable | Count (non-binary) fingerprint generated using Extended Connectivity Fingerprint (ECFP) with radius 2 and 2048 bits. |
| ECFP6        | Bytes     | Nullable | Count (non-binary) fingerprint generated using Extended Connectivity Fingerprint (ECFP) with radius 3 and 2048 bits. |
| FCFP4        | Bytes     | Nullable | Count (non-binary) fingerprint generated using Functional Connectivity Fingerprints (FCFP) with radius 2 and 2048 bits. |
| FCFP6        | Bytes     | Nullable | Count (non-binary) fingerprint generated using Functional Connectivity Fingerprints (FCFP) with radius 3 and 2048 bits. |
| MACCS        | Bytes     | Nullable | FP generated using the Molecular Access System (MACCS).                     |
| RDK          | Bytes     | Nullable | FP generated using the RDKit fingerprint.                                  |
| AVALON       | Bytes     | Nullable | FP generated using the Avalon fingerprint.                                 |
| ATOMPAIR     | Bytes     | Nullable | FP generated using the Atom Pair fingerprint.                               |
| TOPTOR       | Bytes     | Nullable | FP generated using the Topological Torsion fingerprint.                     |
| MW           | Float     | Nullable | Molecular weight of the enumerated DEL rounded to its integer value.        |
| ALOGP        | Float     | Nullable | Calculated LogP of the enumerated DEL rounded to one decimal. Synonymous with ClogP. |

## **3-1- Investigate molecular properties**

**ALOGP:**ALOGP (Atom-based LogP) is a key property in drug discovery that estimates a compound’s lipophilicity, or its ability to dissolve in fats versus water. It predicts the partition coefficient (LogP), which affects drug absorption, permeability, and solubility. A well-balanced LogP value ensures that a drug can efficiently cross biological membranes while remaining soluble enough for proper distribution. In medicinal chemistry, ALOGP is used to optimize drug candidates by improving their bioavailability and reducing potential toxicity.

**Molecular weight:** Molecular weight (MW) is another crucial factor influencing a drug’s pharmacokinetics, including absorption, distribution, metabolism, and excretion. Lower molecular weight compounds tend to have better cell permeability and oral bioavailability, while excessively large molecules may face challenges in diffusion and transport. Lipinski’s Rule of Five recommends an MW below 500 Da for optimal oral drugs, ensuring they can be efficiently absorbed and distributed in the body. Controlling MW helps medicinal chemists design drugs that balance potency, stability, and metabolic efficiency.

In [ ]:
# Create a figure with two subplots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot ALOGP histogram
sns.histplot(data=df_train, x="ALOGP", kde=True, bins=30, ax=axes[0])
axes[0].set_title("Distribution of ALOGP")

# Plot MW histogram
sns.histplot(data=df_train, x="MW", kde=True, bins=30, ax=axes[1])
axes[1].set_title("Distribution of Molecular Weight")

# Adjust layout
plt.tight_layout()
plt.show()

## **3-2- Labels exploration**

Enrichment has been binarized to 0 (not enriched) and 1 (enriched), but we can also look at the raw counts. Note: It is possible for inactive compounds to have non-zero raw count.

In [ ]:
import matplotlib.pyplot as plt

# Plot the histogram
df_train['LABEL'].hist(bins=30, edgecolor='black')

# Add labels and title
plt.xlabel('LABEL')
plt.ylabel('Frequency')
plt.title('Histogram of LABEL')
# Show the plot
plt.show()

In [ ]:
# Plot raw counts and label of enrichment
fig, (ax1, ax2) = plt.subplots(1,2, figsize=(12, 5))

# Label 0 plot
sns.histplot(data=df_train[df_train['LABEL']==0], x="RawCount", bins=50, color='#1f77b4', ax=ax1)
ax1.set_xlim(0, 0.5)
ax1.set_title('Label 0')

# Label 1 plot
sns.histplot(data=df_train[df_train['LABEL']==1], x="RawCount", bins=50, color='#ff7f0e', ax=ax2)
ax2.set_title('Label 1')

plt.tight_layout()
plt.show()

## **3-3- Class imbalance**

This is a very unbalanced data set, a common thing in drug discovery

In [ ]:
# Let's look at how many DEL hits there are
# Calculate the ratio of labels
label_counts = df_train['LABEL'].value_counts(normalize=True)

# Print the ratio of labels
print(label_counts)

## 🟢 **4. Exploring the Screening Dataset**

These are the ASMS compounds we ultimately want to screen. Unlike the DEL training
data they carry a `SMILES` string for every compound, and they have no label.

In [ ]:
# Summary of the test dataset
df_screen.info()

# Columns' names
column_list = df_screen.columns.tolist()
print("\nColumns' names: ", column_list)
print("\n")

# Number of columns and rows
rows, cols = df_train.shape
print(f"Rows: {rows}, Columns: {cols}")

# First 3 samples of the Test dataset
df_screen.head(3)

**Your turn!**

Add code here to check for anomalies in the screening dataset, such as duplicates and null values.

In [ ]:
# Add code here to check for anomalies in the screening dataset, such as duplicates and null values.

---

## 🟢 **5. Visualizing Molecular Structures**
Molecules in the dataset are stored as **SMILES** strings. Using **RDKit**, we will:
- Convert **SMILES to molecule objects**.
- Visualize multiple molecules using `Draw.MolsToGridImage()`.
- Check structural diversity in the dataset.

Understanding molecules visually helps interpret **chemical properties**.

In [ ]:
import random
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

# Randomly select 5 SMILES from df_screen (drop NaNs to avoid errors)
random_smiles = random.sample(df_screen['SMILES'].dropna().tolist(), 5)

# Convert SMILES to RDKit Mol objects
mols = [Chem.MolFromSmiles(smiles) for smiles in random_smiles]

# Draw molecules in a row with SMILES as legends
img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(200, 200), legends=random_smiles)

# Display the image
display(img)

---

## 🟢 **6. What is a SMILES String?**

SMILES (Simplified Molecular Input Line Entry System) is a text representation of a molecule's structure using ASCII characters. It encodes the atoms, bonds, rings, and connectivity in a linear format.

## **How Does SMILES Work?**

- **Atoms** are represented by element symbols (e.g., `C` for carbon, `O` for oxygen).
- **Bonds** are represented as:
  - **Single bond**: `-` *(optional)*
  - **Double bond**: `=`
  - **Triple bond**: `#`
- **Branches** are enclosed in parentheses: `CC(O)C` → represents **isopropanol**.
- **Rings** are denoted by numbers: `c1ccccc1` (Benzene ring).
- **Aromaticity** is denoted by lowercase letters:  
  - `c1ccccc1` (aromatic benzene)  
  - `C1CCCCC1` (cyclohexane)
- **Charged atoms** are denoted with `+` or `-`: `[Na+]`, `[O-]`.

In [ ]:
# Generate Canonical SMILES
# To ensure a standardized (canonical) form of a SMILES string:
from rdkit import Chem
mol = Chem.MolFromSmiles("CCO")  # Ethanol
canonical_smiles = Chem.MolToSmiles(mol, canonical=True)

print("Canonical SMILES:", canonical_smiles)

In [ ]:
# Example:
# Different SMILES for Aspirin (Before Canonicalization)

# Different ways to write Aspirin
smiles_1 = "CC(=O)Oc1ccccc1C(=O)O"   # Common representation
smiles_2 = "O=C(O)c1ccccc1OC(C)=O"   # Rearranged structure

# Convert to canonical SMILES
canonical_1 = Chem.MolToSmiles(Chem.MolFromSmiles(smiles_1), canonical=True)
canonical_2 = Chem.MolToSmiles(Chem.MolFromSmiles(smiles_2), canonical=True)

print("Canonical SMILES 1:", canonical_1)
print("Canonical SMILES 2:", canonical_2)
print("Same molecule?", canonical_1 == canonical_2)

## **Where do we get the SMILES?**

If you have a MOL file (common chemical format) or  SDF file (Structure Data File containing multiple molecules), you can read it and get the SMILES:


### **What is a MOL File?**
A MOL file (Molecular Structure File) is a chemical file format that stores 3D structural information about a molecule, including atoms, bonds, connectivity, and coordinates.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

# Define a simple SMILES (Ethanol)
smiles = "CCO"

# Convert SMILES to an RDKit molecule object
mol = Chem.MolFromSmiles(smiles)

# Save it as a MOL file
mol_file_path = "ethanol.mol"
Chem.MolToMolFile(mol, mol_file_path)

# Display the molecule image
if mol:
    img = Draw.MolToImage(mol)
    display(img)

    # Read and print the MOL file content
    with open(mol_file_path, "r") as file:
        mol_content = file.read()
    print("\nMOL File Content:\n")
    print(mol_content)
else:
    print("Error: Invalid SMILES")

---

## 🟢 **7. Extracting Molecular Fingerprints**

#### **Usage in Drug Discovery**

- **Similarity Searches**: Finding compounds similar to a known active molecule.
- **QSAR Modeling**: Predicting biological activity based on chemical structure.
- **Machine Learning for Drug Discovery**: Used as features in models to classify or predict compound properties.

## **Morgan Fingerprints**

Morgan fingerprints are a type of circular (or radial) molecular fingerprint used in cheminformatics for representing chemical structures numerically. They are generated using the Extended Connectivity Fingerprint (ECFP) algorithm, which encodes molecular features based on the connectivity of atoms within a given radius.

#### **Types of Circular Fingerprints**

#### **1. Extended-Connectivity Fingerprints (ECFP)**
- Encode molecular features by focusing on the structural identity of atoms, such as atomic number, bonds, and connectivity.

#### **2. Functional-Class Fingerprints (FCFP)**
- Encode molecular features by focusing on the **functional roles** of atoms (e.g., hydrogen-bond donors, acceptors, aromaticity, charge).

Reference: Glen, R.C.et all, 2006. Circular fingerprints: flexible molecular descriptors with applications from physical chemistry to ADME.

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.rdMolDescriptors import GetMorganFingerprintAsBitVect
from IPython.display import display

# Define a simple SMILES (Ethanol)
smiles = "CCO"

# Convert SMILES to an RDKit molecule object
mol = Chem.MolFromSmiles(smiles)

# Define fingerprint size (128-bit)
fp_size = 128

# Generate fingerprints
ecfp4 = GetMorganFingerprintAsBitVect(mol, radius=2, nBits=fp_size)  # ECFP4
ecfp6 = GetMorganFingerprintAsBitVect(mol, radius=3, nBits=fp_size)  # ECFP6
fcfp4 = GetMorganFingerprintAsBitVect(mol, radius=2, nBits=fp_size, useFeatures=True)  # FCFP4
fcfp6 = GetMorganFingerprintAsBitVect(mol, radius=3, nBits=fp_size, useFeatures=True)  # FCFP6

# Convert fingerprints to binary strings
def fingerprint_to_binary(fp):
    return "".join(str(int(b)) for b in fp)

# Create a DataFrame for tabular representation of bit vectors
bit_vector_data = {
    "Fingerprint Type": ["ECFP4", "ECFP6", "FCFP4", "FCFP6"],
    "Bit Vector": [list(ecfp4), list(ecfp6), list(fcfp4), list(fcfp6)]
}
df_bit_vectors = pd.DataFrame(bit_vector_data)

# Print the table for bit vectors
print("\nGenerated Fingerprints (128-bit) - Bit Vectors:")
print(df_bit_vectors.to_string(index=False, max_colwidth=100))

# Create a DataFrame for binary strings
binary_data = {
    "Fingerprint Type": ["ECFP4", "ECFP6", "FCFP4", "FCFP6"],
    "Binary String": [
        fingerprint_to_binary(ecfp4),
        fingerprint_to_binary(ecfp6),
        fingerprint_to_binary(fcfp4),
        fingerprint_to_binary(fcfp6),
    ],
}
df_binary = pd.DataFrame(binary_data)

# Print the table for binary strings separately
print("\nGenerated Fingerprints (128-bit) - Binary Representation:")
print(df_binary.to_string(index=False, max_colwidth=100))

# Display the molecule
if mol:
    img = Draw.MolToImage(mol)
    display(img)
else:
    print("Error: Invalid SMILES")

---

## 🟢 **8. Computing Molecular Distance & Similarity**

### **Tanimoto Similarity in Molecular Fingerprints*

- **Mol A and Mol B**: Each molecule is represented as a binary fingerprint, where each bit indicates the presence (`1`) or absence (`0`) of a structural feature.
- **Colored Blocks**:
  - **Red**: Features present in Mol A but not in Mol B.
  - **Blue**: Features present in Mol B but not in Mol A.
  - **Black**: Features shared between Mol A and Mol B (common features).

### **8-1- Convert and Binarize data**

In [ ]:
import numpy as np
import random
import pandas as pd
from rdkit import Chem
from rdkit import DataStructs
from rdkit.Chem import Draw
from IPython.display import display, HTML

# Function to process fingerprint data
def process_data(X, column_name):
    """Stack a fingerprint column into a binarised DataFrame (n_molecules, n_bits)."""
    fps = np.stack(X[column_name].to_numpy())

    # Binarization: Convert values > 0 to 1, else 0
    X_binarized = (fps > 0).astype(int)

    return pd.DataFrame(
        X_binarized,
        columns=[f'{column_name}_{i}' for i in range(X_binarized.shape[1])],
        index=X.index,
    )

# Convert ECFP4 fingerprints to binarized format
df_screen_processed = process_data(df_screen, 'ECFP4')

### **8-2- Check the similarity of two random screening compounds**

In [ ]:
# Randomly select two different indices
n = df_screen.shape[0]
index1, index2 = random.sample(range(n), 2)

# Get selected fingerprints
fp1, fp2 = df_screen_processed.iloc[index1].values, df_screen_processed.iloc[index2].values

# Compute Tanimoto similarity
common = np.logical_and(fp1, fp2).sum()
total = np.logical_or(fp1, fp2).sum()
tanimoto_sim = common / total if total > 0 else 0.0

# Check if SMILES column exists and convert molecules
if 'SMILES' in df_screen.columns:
    mol1, mol2 = Chem.MolFromSmiles(df_screen.iloc[index1]['SMILES']), Chem.MolFromSmiles(df_screen.iloc[index2]['SMILES'])

    # Draw molecules side by side
    img = Draw.MolsToGridImage([mol1, mol2], molsPerRow=2, subImgSize=(200, 200),
                               legends=[f"Molecule {index1}", f"Molecule {index2}"], useSVG=True)
    display(img)

# Display similarity score
display(HTML(f'<h3>Tanimoto Similarity between Molecule {index1} and {index2}: {tanimoto_sim:.4f}</h3>'))

### **8-3- Compute the similarity between screening data and train data:**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def process_data(X, column_name):
    """Stack a fingerprint column into a binarised DataFrame (n_molecules, n_bits)."""
    fps = np.stack(X[column_name].to_numpy())

    # Binarization: Convert values > 0 to 1, else 0
    X_binarized = (fps > 0).astype(int)

    return pd.DataFrame(
        X_binarized,
        columns=[f'{column_name}_{i}' for i in range(X_binarized.shape[1])],
        index=X.index,
    )

# Define fraction of data to sample
sample_frac = 0.05  # 5% of each class

# Sample equal number of positive and negative cases from train dataset
df_train_pos = df_train[df_train['LABEL'] == 1].sample(frac=sample_frac, random_state=42)
df_train_neg = df_train[df_train['LABEL'] == 0].sample(frac=sample_frac, random_state=42)

# Ensure we sample equal number from both classes
min_samples = min(len(df_train_pos), len(df_train_neg))
df_train_pos = df_train_pos.sample(n=min_samples, random_state=42)
df_train_neg = df_train_neg.sample(n=min_samples, random_state=42)

# Combine positive and negative samples to form the final train dataset
df_train_sampled = pd.concat([df_train_pos, df_train_neg]).sample(frac=1, random_state=42)  # Shuffle after combining

# Sample test dataset
df_screen_sampled = df_screen.sample(frac=0.01, random_state=42).copy()

# Convert ECFP4 fingerprints into binarized features
df_screen_fps = process_data(df_screen_sampled, 'ECFP4')
df_train_pos_fps = process_data(df_train_pos, 'ECFP4')
df_train_neg_fps = process_data(df_train_neg, 'ECFP4')

# Function to compute Tanimoto similarity (after binarization)
def compute_tanimoto_similarity(test_fp, train_fps):
    """Compute Tanimoto similarity between a single binarized test fingerprint and a list of binarized train fingerprints."""
    test_fp = test_fp.values.reshape(1, -1)
    train_fps = train_fps.values
    dot_product = np.dot(train_fps, test_fp.T).flatten()

    sum_test = test_fp.sum()
    sum_train = train_fps.sum(axis=1)

    denominator = sum_test + sum_train - dot_product

    # Handle zero division: If denominator is zero, assign similarity as 0.0
    similarity = np.where(denominator == 0, 0.0, dot_product / denominator)

    return similarity.tolist()

# Compute similarity between test and positive (LABEL=1) train data
similarities_pos = [
    compute_tanimoto_similarity(test_fp, df_train_pos_fps)
    for _, test_fp in df_screen_fps.iterrows()
]

# Compute similarity between test and negative (LABEL=0) train data
similarities_neg = [
    compute_tanimoto_similarity(test_fp, df_train_neg_fps)
    for _, test_fp in df_screen_fps.iterrows()
]

# Flatten the lists for visualization
similarities_pos_flat = [sim for sublist in similarities_pos for sim in sublist]
similarities_neg_flat = [sim for sublist in similarities_neg for sim in sublist]

# Plot histogram for test vs. positive class (LABEL=1)
plt.figure(figsize=(8, 5))
sns.histplot(similarities_pos_flat, bins=30, kde=False, color='green')
plt.yscale("log")
plt.xlabel("Tanimoto Similarity")
plt.ylabel("Frequency")
plt.title("Histogram of Tanimoto Similarity (Test vs LABEL=1 - Positive Class)")
plt.show()

# Plot histogram for test vs. negative class (LABEL=0)
plt.figure(figsize=(8, 5))
sns.histplot(similarities_neg_flat, bins=30, kde=False, color='red')
plt.yscale("log")
plt.xlabel("Tanimoto Similarity")
plt.ylabel("Frequency")
plt.title("Histogram of Tanimoto Similarity (Test vs LABEL=0 - Negative Class)")
plt.show()

---

## 🟢 **9. Drawing UMAP Visualizations**
Universal Manifold Approximation (UMAP) is a fast, scalable dimensionality reduction method that preserves both local and global data structure. It is widely used for clustering, classification, and visualization in fields like bioinformatics and machine learning.

**Key Features:**
- Preserves structure better than PCA and t-SNE
- Scales efficiently to large datasets
- Faster than t-SNE with high-quality embeddings
- Versatile for images, text, and biological data

**UMAP helps in:**
- Reducing high-dimensional fingerprint data to 2D.
- Visualizing clusters of similar molecules.
- Identifying outliers and relationships.

In [ ]:
import numpy as np
import pandas as pd
import umap
import seaborn as sns
import matplotlib.pyplot as plt

# Select 1000 samples from each LABEL (balanced training set)
df_train_0 = df_train[df_train['LABEL'] == 0].sample(n=1000, random_state=42)
df_train_1 = df_train[df_train['LABEL'] == 1].sample(n=1000, random_state=42)

# Combine to form a balanced training dataset
df_train_balanced = pd.concat([df_train_0, df_train_1])

# Randomly select 2000 samples from the screening data
df_screen_sampled = df_screen.sample(n=2000, random_state=42)

# Stack the ECFP4 arrays and binarise them - the jaccard metric below expects
# presence/absence, not counts.
X_train = (np.stack(df_train_balanced["ECFP4"].to_numpy()) > 0).astype(np.uint8)
X_test = (np.stack(df_screen_sampled["ECFP4"].to_numpy()) > 0).astype(np.uint8)

# Combine train and test before fitting UMAP
X_combined = np.vstack([X_train, X_test])

# Prepare UMAP and fit on the full dataset
umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.1,
    metric="jaccard",
    random_state=42
)
embedding_combined = umap_model.fit_transform(X_combined)

# Convert to DataFrame
df_combined = pd.DataFrame(embedding_combined, columns=['UMAP1', 'UMAP2'])

# Separate train and test embeddings
df_train_emb = df_combined.iloc[:len(X_train)].copy()
df_screen_emb = df_combined.iloc[len(X_train):].copy()

# Assign labels for train data
df_train_emb['LABEL'] = df_train_balanced['LABEL'].values  # Keep balanced labels

# Plot train data first (colored by LABEL: 0=Red, 1=Blue)
plt.figure(figsize=(8,6))
sns.scatterplot(data=df_train_emb, x='UMAP1', y='UMAP2', hue='LABEL', s=8, palette={0: 'red', 1: 'blue'}, alpha=0.7)

# Plot test data separately in green
sns.scatterplot(data=df_screen_emb, x='UMAP1', y='UMAP2', color='green', s=8, label='Screening Data', alpha=0.5)

# Final plot settings
plt.title('UMAP Projection of Train & Screening Data (Train: Red/Blue, Screen: Green)')
plt.legend()
plt.show()

---

## 🟢 **10. Feature Selection**
It is possible that not all fingerprint bits or descriptors contribute meaningfully to the model. Some features may:

- **Have very low variance** → These features are almost constant across molecules and don’t provide useful information.
- **Are redundant** → Some features are highly correlated with others and can be removed without losing information.

### Why Compute Feature Variance?
- **Remove Low-Variance Features** → These features don’t change much and won’t help
the model.
- **Select the Most Relevant Features** → Features with high variance tend to capture useful molecular properties.

The threshold for **low variance** depends on the **scale of the data** and the **domain of the problem**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def process_data(X, column_name):
    """Stack a fingerprint column into a DataFrame of counts (n_molecules, n_bits)."""
    fps = np.stack(X[column_name].to_numpy())
    return pd.DataFrame(
        fps,
        columns=[f'{column_name}_{i}' for i in range(fps.shape[1])],
        index=X.index,
    )

# List of fingerprint column names
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4','FCFP6', 'MACCS','RDK','AVALON','ATOMPAIR','TOPTOR']

df_screen_sampled = df_screen.sample(n=250, random_state=42)

# Process each fingerprint column separately and compute variance
feature_variances = {}
for col in fingerprint_columns:
    processed_fps = process_data(df_screen_sampled, col)  # Process the column
    feature_variances[col] = processed_fps.var().mean()  # Compute the average variance of its bits

# Convert the variance dictionary to a DataFrame for visualization
feature_variance_df = pd.DataFrame(feature_variances.items(), columns=['Fingerprint Type', 'Variance'])

# Plot feature variance across fingerprint types
plt.figure(figsize=(8, 5))
sns.barplot(x='Fingerprint Type', y='Variance', data=feature_variance_df,
            hue='Fingerprint Type', palette='viridis', legend=False)
plt.ylabel("Average Variance Across Bits")
plt.xlabel("Fingerprint Type")
plt.title("Variance of Molecular Fingerprint Features")
plt.xticks(rotation=45)
plt.show()

# Apply variance threshold to filter out low-variance fingerprints
threshold = 0.01  # Define the minimum variance threshold
selected_fingerprints = feature_variance_df[feature_variance_df['Variance'] > threshold]['Fingerprint Type'].tolist()

# Retain only informative fingerprint types
df_screen_filtered = df_screen_sampled[selected_fingerprints]

# Print summary
print(f"Original Features: {len(fingerprint_columns)}")
print(f"Features Retained After Variance Filtering: {len(selected_fingerprints)}")
print("\nRetained Features:\n", selected_fingerprints)

---

## 🟢 **11. Sparcity**

## 📌 Understanding Low Variance and Sparsity in Feature Selection

Sparsity can change the game in feature selection:
- Some features with high sparsity may correlate strongly with important molecular properties (e.g., bioactivity, toxicity).
- Variance filtering alone is not enough—we also need feature importance methods (e.g., Random Forest, SHAP values).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def process_data(X, column_name):
    """Stack a fingerprint column into a DataFrame of counts (n_molecules, n_bits)."""
    fps = np.stack(X[column_name].to_numpy())
    return pd.DataFrame(
        fps,
        columns=[f'{column_name}_{i}' for i in range(fps.shape[1])],
        index=X.index,
    )

# List of fingerprint column names
fingerprint_columns = ['ECFP4', 'ECFP6', 'FCFP4', 'FCFP6', 'MACCS', 'RDK', 'AVALON', 'ATOMPAIR', 'TOPTOR']

# Sample 250 molecules from df_screen
df_screen_sampled = df_screen.sample(n=250, random_state=42)

# Process each fingerprint column separately and compute sparsity
feature_sparsity = {}
for col in fingerprint_columns:
    processed_fps = process_data(df_screen_sampled, col)  # Process the column
    sparsity = (processed_fps == 0).mean().mean()  # Compute sparsity as the proportion of zeros
    feature_sparsity[col] = sparsity

# Convert the sparsity dictionary to a DataFrame for visualization
feature_sparsity_df = pd.DataFrame(feature_sparsity.items(), columns=['Fingerprint Type', 'Sparsity'])

# Plot feature sparsity across fingerprint types
plt.figure(figsize=(8, 5))
sns.barplot(x='Fingerprint Type', y='Sparsity', data=feature_sparsity_df,
            hue='Fingerprint Type', palette='magma', legend=False)
plt.ylabel("Proportion of Zeros (Sparsity)")
plt.xlabel("Fingerprint Type")
plt.title("Sparsity of Molecular Fingerprint Features")
plt.xticks(rotation=45)
plt.ylim(0, 1)  # Sparsity is between 0 and 1
plt.show()

# Print summary
print(f"Computed sparsity for {len(fingerprint_columns)} fingerprint types.")
print("\nFeature Sparsity:\n", feature_sparsity_df)

---

# ✅ **Next Steps**
Now that we have a **clean dataset**, we are ready to move on to **Machine Learning!** 🎯

---

## ⏱️ How long did that take?

Wall-clock time for the whole notebook, then every cell in the order it ran, and the slowest
five pulled out. Useful for planning a session, and for spotting a cell that is slower than
it looks.

To watch the timings live instead of waiting for this summary, set `ECHO_CELL_TIME = True` in
the first cell — each cell then prints its own time underneath its output.

In [ ]:
elapsed = _time.time() - NOTEBOOK_STARTED
minutes, seconds = divmod(elapsed, 60)

print(f"Total run time : {int(minutes)} min {seconds:04.1f} s")
print(f"Cells executed : {len(CELL_TIMES)}")

if CELL_TIMES:
    measured = sum(t for t, _ in CELL_TIMES)
    print(f"Time in cells  : {measured:.1f} s "
          f"({measured / elapsed:.0%} of the total; the rest is start-up and idle time)")

    # every cell, in the order it ran
    print(f"\n{'cell':>4} {'seconds':>9} {'share':>7}  first line")
    print("-" * 78)
    for n, (taken, first_line) in enumerate(CELL_TIMES, start=1):
        share = taken / measured if measured else 0
        marker = " <-- slow" if taken >= 5 else ""
        print(f"{n:>4} {taken:>9.2f} {share:>6.1%}  {first_line}{marker}")

    print(f"\nSlowest five:")
    for taken, first_line in sorted(CELL_TIMES, reverse=True)[:5]:
        print(f"  {taken:6.1f}s  {first_line}")

print(f"\nMeasured on whatever machine ran this. Colab is usually slower than a laptop,")
print("so treat these as a guide rather than a promise.")
print("Set ECHO_CELL_TIME = True in the first cell to see each cell timed as it runs.")